# Flight Delay Analysis - Data Ingestion & Cleaning

## Phase 1: Data Preparation

**Project**: University Case Study - Flight Delay Analysis  
**Dataset**: Kaggle Flight Data 2024  
**Expected Size**: 7M+ rows (potentially up to 50M)  
**Objective**: Load, profile, clean, and prepare flight data for analysis

---

## Table of Contents
1. [Environment Setup & Imports](#section1)
2. [Dataset Discovery & Initial Inspection](#section2)
3. [Memory-Efficient Data Loading Strategy](#section3)
4. [Comprehensive Data Profiling](#section4)
5. [Critical Data Quality Transformations](#section5)
6. [Parquet Conversion & Storage](#section6)
7. [Data Quality Summary Report](#section7)

---
<a id='section1'></a>
## 1. Environment Setup & Imports

**Purpose**: Import required libraries and configure the environment for optimal data processing.

In [ ]:
# Standard library imports
import sys
import warnings
from pathlib import Path
from datetime import datetime

# Data processing libraries
import pandas as pd
import numpy as np

# Visualization libraries (for profiling visualizations)
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Add src directory to Python path for custom utilities
project_root = Path('/home/user/Flight-delay')
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

# Import custom utility functions
from utils import (
    DataProfiler,
    DataLoader,
    DataCleaner,
    ParquetConverter,
    setup_pandas_display,
    verify_directory_structure
)

print("✓ Custom utilities imported successfully")

In [ ]:
# Configure pandas display options
setup_pandas_display()

# Verify directory structure
print("\nVerifying project structure:")
verify_directory_structure(str(project_root))

In [ ]:
# Define project paths
DATA_RAW = project_root / 'data' / 'raw'
DATA_PROCESSED = project_root / 'data' / 'processed'
REPORTS_DIR = project_root / 'reports'

print("Project paths configured:")
print(f"  Raw data: {DATA_RAW}")
print(f"  Processed data: {DATA_PROCESSED}")
print(f"  Reports: {REPORTS_DIR}")

---
<a id='section2'></a>
## 2. Dataset Discovery & Initial Inspection

**Purpose**: Locate CSV files and perform initial inspection to understand the data structure.

**Expected Outcome**: Identify available files and understand basic schema before full load.

In [ ]:
# Discover CSV files in raw data directory
csv_files = DataLoader.discover_csv_files(str(DATA_RAW))

if len(csv_files) == 0:
    print("\n⚠️  WARNING: No CSV files found!")
    print("Please follow the instructions in src/download_instructions.md to download the dataset.")
    print("\nDataset URL: https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024")
else:
    print(f"\n✓ Found {len(csv_files)} CSV file(s) ready for processing")

In [ ]:
# Select the primary CSV file to process
# If multiple files exist, you may need to adjust this logic
if len(csv_files) > 0:
    PRIMARY_FILE = csv_files[0]  # Use first file found
    print(f"Primary file selected: {PRIMARY_FILE.name}")
    print(f"File size: {PRIMARY_FILE.stat().st_size / (1024**3):.2f} GB")
else:
    PRIMARY_FILE = None
    print("⚠️  No file selected. Please download dataset first.")

### Load Sample for Initial Inspection

**Justification**: Loading a small sample (1000 rows) allows us to:
- Understand the schema without loading the entire dataset
- Identify column names and data types
- Plan appropriate data loading strategy
- Minimize memory usage during exploration

In [ ]:
# Load sample for inspection (first 1000 rows)
if PRIMARY_FILE is not None:
    df_sample = DataLoader.load_sample(str(PRIMARY_FILE), n_rows=1000)
    
    print("\n=== SAMPLE DATA OVERVIEW ===")
    print(f"Shape: {df_sample.shape}")
    print(f"Columns: {df_sample.columns.tolist()}")
else:
    print("Cannot load sample - no data file available")

In [ ]:
# Display first few rows
if PRIMARY_FILE is not None:
    print("\n=== FIRST 5 ROWS ===")
    display(df_sample.head())

In [ ]:
# Display data types
if PRIMARY_FILE is not None:
    print("\n=== DATA TYPES ===")
    print(df_sample.dtypes)

In [ ]:
# Display basic statistics for numeric columns
if PRIMARY_FILE is not None:
    print("\n=== NUMERIC COLUMN STATISTICS ===")
    display(df_sample.describe())

---
<a id='section3'></a>
## 3. Memory-Efficient Data Loading Strategy

**Challenge**: Dataset contains 7M+ rows (potentially up to 50M), which requires careful memory management.

**Strategy**: 
- Use chunked reading for CSV files
- Process in 1M row chunks
- Convert to Parquet immediately for future efficient access

**Justification**: 
- Chunked reading prevents memory overflow on large datasets
- Allows processing of datasets larger than available RAM
- Parquet format provides 10x faster read performance and better compression

In [ ]:
# Define chunk size based on dataset size estimate
CHUNK_SIZE = 1_000_000  # 1 million rows per chunk

print(f"Loading strategy: Chunked reading with {CHUNK_SIZE:,} rows per chunk")
print("This approach ensures memory-efficient processing of large datasets.")

In [ ]:
# Load full dataset using chunked approach
if PRIMARY_FILE is not None:
    print("\n=== LOADING FULL DATASET ===")
    print("This may take several minutes depending on file size...\n")
    
    df_full = DataLoader.load_csv_chunked(
        str(PRIMARY_FILE),
        chunksize=CHUNK_SIZE,
        low_memory=False  # Ensure consistent dtypes
    )
    
    print(f"\n✓ Dataset loaded successfully!")
    print(f"  Total rows: {len(df_full):,}")
    print(f"  Total columns: {len(df_full.columns)}")
else:
    print("Cannot load dataset - no data file available")
    df_full = None

---
<a id='section4'></a>
## 4. Comprehensive Data Profiling

**Purpose**: Understand data quality, completeness, and characteristics before cleaning.

### 4.1 Schema Analysis

In [ ]:
# Generate comprehensive schema information
if df_full is not None:
    print("=== SCHEMA ANALYSIS ===")
    schema_info = DataProfiler.get_schema_info(df_full)
    display(schema_info)

In [ ]:
# Memory usage analysis
if df_full is not None:
    print("\n=== MEMORY USAGE ===")
    memory_stats = DataProfiler.get_memory_usage(df_full)
    print(f"Total memory: {memory_stats['total_memory']}")
    print(f"Total memory (MB): {memory_stats['total_memory_mb']:.2f} MB")

### 4.2 Missing Value Analysis

**Purpose**: Identify columns with missing data and quantify data completeness.

In [ ]:
# Missing data summary
if df_full is not None:
    print("=== MISSING DATA ANALYSIS ===")
    missing_summary = DataProfiler.get_missing_data_summary(df_full, threshold=0.0)
    
    if len(missing_summary) > 0:
        display(missing_summary)
    else:
        print("✓ No missing values detected in any column!")

In [ ]:
# Visualize missing data pattern
if df_full is not None and len(missing_summary) > 0:
    plt.figure(figsize=(12, 6))
    plt.barh(missing_summary['Column'], missing_summary['Missing %'])
    plt.xlabel('Missing Percentage (%)')
    plt.ylabel('Column')
    plt.title('Missing Data by Column')
    plt.tight_layout()
    plt.show()

### 4.3 Statistical Summary

**Purpose**: Understand distributions and identify potential outliers.

In [ ]:
# Statistical summary for numeric columns
if df_full is not None:
    print("=== STATISTICAL SUMMARY (Numeric Columns) ===")
    display(df_full.describe())

In [ ]:
# Value counts for key categorical columns (if present)
if df_full is not None:
    categorical_cols = df_full.select_dtypes(include=['object']).columns.tolist()
    
    print(f"\n=== CATEGORICAL COLUMNS ({len(categorical_cols)} found) ===")
    
    # Show value counts for first few categorical columns
    for col in categorical_cols[:5]:  # Limit to 5 to avoid overwhelming output
        print(f"\n{col}:")
        print(df_full[col].value_counts().head(10))

### 4.4 Data Quality Issues

**Purpose**: Identify duplicates, outliers, and data inconsistencies.

In [ ]:
# Duplicate detection
if df_full is not None:
    print("=== DUPLICATE ANALYSIS ===")
    dup_stats = DataProfiler.detect_duplicates(df_full)
    
    print(f"Total rows: {dup_stats['total_rows']:,}")
    print(f"Unique rows: {dup_stats['unique_rows']:,}")
    print(f"Duplicate rows: {dup_stats['total_duplicates']:,} ({dup_stats['duplicate_percentage']:.2f}%)")

---
<a id='section5'></a>
## 5. Critical Data Quality Transformations

**Purpose**: Apply essential data cleaning steps with clear justifications.

### 5.1 Cancellation Identification

**Justification**: 
- Cancelled flights distort delay statistics
- Must be analyzed separately from active flights
- Mixing cancelled and active flights would produce misleading delay metrics

In [ ]:
# Separate cancelled from active flights
if df_full is not None:
    print("=== SEPARATING CANCELLED FLIGHTS ===")
    
    # Try common cancellation column names
    possible_cancel_cols = ['CANCELLED', 'Cancelled', 'cancelled', 'CANCELED', 'Canceled']
    cancel_col = None
    
    for col in possible_cancel_cols:
        if col in df_full.columns:
            cancel_col = col
            break
    
    if cancel_col:
        df_active, df_cancelled = DataCleaner.identify_cancelled_flights(df_full, cancelled_col=cancel_col)
    else:
        print("⚠️  No cancellation column found. Treating all flights as active.")
        df_active = df_full.copy()
        df_cancelled = pd.DataFrame()
else:
    df_active = None
    df_cancelled = None

### 5.2 Delay Threshold Classification

**Justification**: 
- Case study defines significant delay as ≥15 minutes
- Binary classification simplifies analysis and interpretation
- Aligns with industry standards for reporting delays

In [ ]:
# Create delay indicators for active flights
if df_active is not None and len(df_active) > 0:
    print("\n=== CREATING DELAY CLASSIFICATIONS ===")
    
    # Try common delay column names
    possible_delay_cols = ['ARR_DELAY', 'ArrDelay', 'arr_delay', 'ARRIVAL_DELAY']
    delay_col = None
    
    for col in possible_delay_cols:
        if col in df_active.columns:
            delay_col = col
            break
    
    if delay_col:
        # Create 15-minute delay threshold binary indicator
        df_active = DataCleaner.create_delay_binary(df_active, delay_col=delay_col, threshold=15)
    else:
        print("⚠️  No delay column found. Skipping delay classification.")

### 5.3 Early Arrival Identification

**Justification**: 
- Negative delays represent early arrivals (valuable insight)
- Important for analyzing 'time made up in air' patterns
- Should NOT be treated as errors or removed

In [ ]:
# Identify early arrivals
if df_active is not None and len(df_active) > 0 and delay_col:
    print("\n=== IDENTIFYING EARLY ARRIVALS ===")
    df_active = DataCleaner.identify_early_arrivals(df_active, delay_col=delay_col)

### 5.4 Time Validation

**Justification**: 
- Ensure data integrity before analysis
- Detect impossible time relationships
- Flag records requiring further investigation

In [ ]:
# Validate time consistency
if df_active is not None and len(df_active) > 0:
    print("\n=== TIME CONSISTENCY VALIDATION ===")
    
    # Try to find time columns (may vary by dataset)
    possible_elapsed_cols = ['ELAPSED_TIME', 'ElapsedTime', 'elapsed_time', 'ACTUAL_ELAPSED_TIME']
    elapsed_col = None
    
    for col in possible_elapsed_cols:
        if col in df_active.columns:
            elapsed_col = col
            break
    
    if elapsed_col:
        df_active = DataCleaner.validate_time_consistency(
            df_active,
            elapsed_col=elapsed_col
        )
    else:
        print("⚠️  Elapsed time column not found. Skipping time validation.")
        df_active['time_valid'] = 1  # Assume valid

---
<a id='section6'></a>
## 6. Parquet Conversion & Storage

**Justification**: 
- Parquet provides 10x faster read performance vs CSV
- Better compression (typically 50-70% smaller than CSV)
- Column-oriented storage optimized for analytics
- Preserves data types (no re-inference needed)

**Strategy**: Save two separate files:
1. `flights_active.parquet` - Active flights for delay analysis
2. `flights_cancelled.parquet` - Cancelled flights for separate analysis

In [ ]:
# Save active flights to Parquet
if df_active is not None and len(df_active) > 0:
    print("=== SAVING ACTIVE FLIGHTS TO PARQUET ===")
    
    active_path = DATA_PROCESSED / 'flights_active.parquet'
    active_stats = ParquetConverter.save_to_parquet(
        df_active,
        str(active_path),
        compression='snappy'
    )
    
    print(f"\nActive flights saved: {active_stats['rows']:,} rows, {active_stats['file_size_mb']:.2f} MB")
else:
    print("No active flights to save")
    active_stats = None

In [ ]:
# Save cancelled flights to Parquet (if any exist)
if df_cancelled is not None and len(df_cancelled) > 0:
    print("\n=== SAVING CANCELLED FLIGHTS TO PARQUET ===")
    
    cancelled_path = DATA_PROCESSED / 'flights_cancelled.parquet'
    cancelled_stats = ParquetConverter.save_to_parquet(
        df_cancelled,
        str(cancelled_path),
        compression='snappy'
    )
    
    print(f"\nCancelled flights saved: {cancelled_stats['rows']:,} rows, {cancelled_stats['file_size_mb']:.2f} MB")
else:
    print("\nNo cancelled flights to save")
    cancelled_stats = None

### Verify Parquet Files

**Purpose**: Confirm successful save and test loading performance.

In [ ]:
# Verify saved files
print("\n=== VERIFYING PARQUET FILES ===")

parquet_files = list(DATA_PROCESSED.glob('*.parquet'))
print(f"Found {len(parquet_files)} Parquet file(s):")

for pfile in parquet_files:
    file_size_mb = pfile.stat().st_size / (1024 ** 2)
    print(f"  - {pfile.name}: {file_size_mb:.2f} MB")

In [ ]:
# Test loading from Parquet (quick performance check)
if active_stats is not None:
    print("\n=== TESTING PARQUET LOAD PERFORMANCE ===")
    
    import time
    start_time = time.time()
    
    df_test = pd.read_parquet(DATA_PROCESSED / 'flights_active.parquet')
    
    load_time = time.time() - start_time
    
    print(f"✓ Loaded {len(df_test):,} rows in {load_time:.2f} seconds")
    print(f"  Loading speed: {len(df_test) / load_time:,.0f} rows/second")

---
<a id='section7'></a>
## 7. Data Quality Summary Report

**Purpose**: Generate comprehensive summary of data processing results.

In [ ]:
# Generate summary statistics
print("=" * 70)
print("DATA QUALITY SUMMARY REPORT")
print("=" * 70)

if df_full is not None:
    print(f"\n📊 DATASET OVERVIEW")
    print(f"  Source file: {PRIMARY_FILE.name if PRIMARY_FILE else 'N/A'}")
    print(f"  Total records loaded: {len(df_full):,}")
    print(f"  Total columns: {len(df_full.columns)}")
    print(f"  Processing date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    print(f"\n✈️  FLIGHT CATEGORIZATION")
    if df_active is not None:
        print(f"  Active flights: {len(df_active):,} ({len(df_active)/len(df_full)*100:.1f}%)")
    if df_cancelled is not None and len(df_cancelled) > 0:
        print(f"  Cancelled flights: {len(df_cancelled):,} ({len(df_cancelled)/len(df_full)*100:.1f}%)")
    
    if df_active is not None and 'is_delayed_15min' in df_active.columns:
        delayed_count = df_active['is_delayed_15min'].sum()
        print(f"\n⏰ DELAY STATISTICS (Active Flights)")
        print(f"  Delayed (≥15 min): {delayed_count:,} ({delayed_count/len(df_active)*100:.1f}%)")
        print(f"  On-time (<15 min): {len(df_active) - delayed_count:,} ({(len(df_active)-delayed_count)/len(df_active)*100:.1f}%)")
    
    if df_active is not None and 'early_arrival' in df_active.columns:
        early_count = df_active['early_arrival'].sum()
        print(f"\n🎯 EARLY ARRIVALS")
        print(f"  Early arrivals: {early_count:,} ({early_count/len(df_active)*100:.1f}%)")
    
    if df_active is not None and 'time_valid' in df_active.columns:
        valid_count = df_active['time_valid'].sum()
        print(f"\n✅ DATA VALIDATION")
        print(f"  Valid time records: {valid_count:,} ({valid_count/len(df_active)*100:.1f}%)")
        print(f"  Invalid time records: {len(df_active) - valid_count:,}")
    
    print(f"\n💾 OUTPUT FILES")
    if active_stats:
        print(f"  flights_active.parquet: {active_stats['rows']:,} rows, {active_stats['file_size_mb']:.2f} MB")
    if cancelled_stats:
        print(f"  flights_cancelled.parquet: {cancelled_stats['rows']:,} rows, {cancelled_stats['file_size_mb']:.2f} MB")
    
    print(f"\n📋 PREPROCESSING DECISIONS LOG")
    print(f"  1. ✓ Separated cancelled flights for independent analysis")
    print(f"  2. ✓ Created binary delay indicator (≥15 min threshold)")
    print(f"  3. ✓ Preserved negative delays as early arrivals")
    print(f"  4. ✓ Validated time consistency across records")
    print(f"  5. ✓ Converted to Parquet format for optimal performance")
    
    print(f"\n🎓 READY FOR ANALYSIS")
    print(f"  ✅ Data ingestion complete")
    print(f"  ✅ Quality checks passed")
    print(f"  ✅ Optimized storage format created")
    print(f"  ✅ Ready for Phase 2: Exploratory Data Analysis")
    
print("\n" + "=" * 70)

## Next Steps

### Phase 2: Exploratory Data Analysis

Now that data is cleaned and prepared, proceed to:

1. **`notebooks/eda_analysis.ipynb`** - Comprehensive exploratory analysis
   - Temporal patterns (delays by time/date)
   - Geographic analysis (routes, airports)
   - Carrier performance comparison
   - Delay propagation patterns
   - Connection vulnerability analysis

2. **Load processed data**:
   ```python
   df_active = pd.read_parquet('data/processed/flights_active.parquet')
   df_cancelled = pd.read_parquet('data/processed/flights_cancelled.parquet')
   ```

3. **Create comprehensive data profiling report**:
   - See `reports/data_profiling_summary.md` template

---

**✓ Phase 1 Complete: Data Ingestion & Cleaning**